In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm 
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats
import random
from scipy import stats
from scipy.stats import fisher_exact
from scipy.stats.contingency import expected_freq
import glob
sns.set(font_scale=1,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})
from uncertainties import ufloat
import scipy.optimize as opt 

In [ ]:
ThT_WT  = pd.read_pickle('ThT_96.pkl')

In [ ]:
df = pd.read_excel("/Users/aelazar/Library/CloudStorage/OneDrive-SharedLibraries-St.JudeChildren'sResearchHospital/Trivedi, Vikas - Assaf-Vikas/IDR_dSAMs/ThT/2025_05_03_NGFR_Orthologs.xlsx",
              sheet_name='Day3',
              header=0)
blank = 9848.166667

In [ ]:
l =[]
for col in df.columns:
    spe   = col.split('_')[1]
    if len (col.split('_'))==3:
        ele = 'P'
    else:
        ele='WT'
    for ind in df.index:
        l.append({'ThT':df.loc[ind,col]-blank,'spe':spe,'ele':ele,'non_blank':df.loc[ind,col]})

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))
plot_df = pd.DataFrame(l)
sns.barplot(x='spe',y='non_blank',data=plot_df,hue='ele',fill=False)
plt.axhline(blank)
plt.yscale('log')

In [ ]:
fig, ax = plt.subplots(figsize=(2, 3))
sns.pointplot(x='ele',y='non_blank',data=plot_df
              ,hue='spe',errorbar='se',dodge=True,
              scale=.75,palette=['#8964AA','#1072BB','#3BB97F'],log_scale=(False,True),linestyles='--')
plt.axhline(blank,ls='--',color='k')
plt.savefig('ThT_orthologs_NGFR.svg',bbox_inches='tight',transparent=True)

In [ ]:
names = ['NGFR_CHICK','NGFR_CHICK_MUT','NGFR_LEPOC','NGFR_LEPOC_MUT','NGFR_BOVIN','NGFR_BOVIN_MUT',
         'TNFRSF13C_CHICK','TNFRSF13C_CHICK_MUT','TNFRSF13C_FELCA','TNFRSF13C_FELCA_MUT','TNFRSF13C_MACMU',
         'TNFRSF13C_MACMU_MUT']
l = []
for n in enumerate([i.split('_') for i in names]):
    
    if n[0]+1<10:
        well = '0%d'%int(n[0]+1)
    else:
        well = '%d'%int(n[0]+1) 
    name = n[1][0]
    spe = n[1][1]
    if n[0] %2 ==1 :
        ele = 'mut'
    else:
        ele = 'WT'
    l.append({'well':well,'name':name,'spe':spe,'ele':ele})

In [ ]:
palte_df = pd.DataFrame(l)

In [ ]:
# H is balnk 

df = pd.read_excel("/Users/aelazar/Library/CloudStorage/OneDrive-SharedLibraries-St.JudeChildren'sResearchHospital/Trivedi, Vikas - Assaf-Vikas/IDR_dSAMs/ThT/20250512_Ortholog_ThT_2.xlsx",
              sheet_name='Table End point',
              header=10)
blanks = df[(df['Unnamed: 0'].str.find('H'))!=-1].iloc[:,2:].mean().reset_index()
df = df.drop(index=df[(df['Unnamed: 0'].str.find('H'))!=-1].index)
df = df.drop(index=df[(df['Unnamed: 0'].str.find('G'))!=-1].index)
l = []
for i in df.index:
    well = df.loc[i,'Unnamed: 0'][1:]
    
    for nm in [465,470,475,480,485,490,495,500,505,510,515,520,525,530,535,540,545,550,555]:
        val = df.loc[i,nm]
        l.append({'name':well,'nm':nm,'val':val})#blanks[blanks['index']==465][0].values[0]
for val in ThT_WT[ThT_WT.gene_name=='TNFRSF13C'].ThT:
    l.append({'name':'TNFRSF13C_human','nm':490,'val':val})
for val in ThT_WT[ThT_WT.gene_name=='NGFR'].ThT:
    l.append({'name':'NGFR_human','nm':490,'val':val})   

In [ ]:
df_plot = pd.DataFrame(l)
df_plot

In [ ]:
df_plot['human_ortholog'] = df_plot.name.map(dict(zip(palte_df.well,palte_df.name)))
df_plot['spe'] = df_plot.name.map(dict(zip(palte_df.well,palte_df.spe)))
df_plot['ele'] = df_plot.name.map(dict(zip(palte_df.well,palte_df.ele)))
df_plot.loc[df_plot[df_plot.name=='TNFRSF13C_human'].index,'human_ortholog'] = 'TNFRSF13C'
df_plot.loc[df_plot[df_plot.name=='TNFRSF13C_human'].index,'spe'] = 'human'
df_plot.loc[df_plot[df_plot.name=='TNFRSF13C_human'].index,'ele'] = 'WT'
df_plot.loc[df_plot[df_plot.name=='NGFR_human'].index,'human_ortholog'] = 'NGFR'
df_plot.loc[df_plot[df_plot.name=='NGFR_human'].index,'spe'] = 'human'
df_plot.loc[df_plot[df_plot.name=='NGFR_human'].index,'ele'] = 'WT'

In [ ]:
sns.lineplot(x='nm',y='val',data=df_plot[(df_plot.human_ortholog=='TNFRSF13C')&(df_plot.spe=='CHICK')]
             ,hue='ele',err_style='bars',hue_order=['WT','mut'])
#plt.yscale('log')

In [ ]:
sns.barplot(x='spe',y='val',data=df_plot[(df_plot.nm==490)&(df_plot.human_ortholog=='NGFR')],hue='ele')
plt.yscale('log')

In [ ]:
set(df_plot[(df_plot.human_ortholog=='TNFRSF13C')].spe)

In [ ]:
blanks

In [ ]:
fig, ax = plt.subplots(figsize=(2, 3))
sns.pointplot(x='ele',y='val',data=df_plot[(df_plot.nm==490)&(df_plot.human_ortholog=='TNFRSF13C')]
              ,hue='spe',errorbar='se',dodge=True,
              scale=.75,hue_order=['CHICK','FELCA','human'],palette=['#8964AA','#1072BB','#3BB97F','#6E778F'],log_scale=(False,True),linestyles='--')
plt.axhline(166.500000,ls='--',color='k')
plt.savefig('ThT_orthologs_TNFRSF13C.svg',bbox_inches='tight',transparent=True)

In [ ]:
fig, ax = plt.subplots(figsize=(2, 3))
sns.pointplot(x='ele',y='val',data=df_plot[(df_plot.nm==490)&(df_plot.human_ortholog=='NGFR')&(df_plot.spe!='BOVIN')]
              ,hue='spe',errorbar='se',dodge=True,
              scale=.75,hue_order=['CHICK','LEPOC','human'],palette=['#8964AA','#3470B5','#414042'],log_scale=(False,True),linestyles='--')
plt.axhline(166,ls='--',color='k')

plt.savefig('ThT_orthologs_NGFR.svg',bbox_inches='tight',transparent=True)

In [ ]:
set(df_plot.spe)

In [ ]:
b = []
for i in blanks.index:
   b.append({'val':blanks.loc[i,0],'nm':blanks.loc[i,'index']}) 
blanks_df = pd.DataFrame(b)

In [ ]:
for spe in ['FELCA','CHICK','MACMU']:
    fig, ax = plt.subplots(figsize=(3, 3))
    sns.lineplot(x='nm',y='val',data=df_plot[(df_plot.human_ortholog=='TNFRSF13C')&(df_plot.spe==spe)]
                 ,hue='ele',err_style='band',hue_order=['WT','mut'],palette=['#be1e2d','#414042'],zorder=10)
    sns.lineplot(x='nm',y='val',data=blanks_df,color='#BDBCBE',zorder=0)
    plt.title('%s'%spe)
    plt.savefig('TNFRSF13C_%s_spectra.svg'%spe,bbox_inches='tight',transparent=True)
    #plt.yscale('log')

In [ ]:
for spe in ['LEPOC','CHICK']:
    fig, ax = plt.subplots(figsize=(3, 3))
    sns.lineplot(x='nm',y='val',data=df_plot[(df_plot.human_ortholog=='NGFR')&(df_plot.spe==spe)]
                 ,hue='ele',err_style='band',hue_order=['WT','mut'],palette=['#be1e2d','#414042'],zorder=10)
    sns.lineplot(x='nm',y='val',data=blanks_df,color='#BDBCBE',zorder=0)
    plt.title('%s'%spe)
    plt.savefig('NGFR_%s_spectra.svg'%spe,bbox_inches='tight',transparent=True)
    #plt.yscale('log')

In [ ]:
blanks_df